In [1]:
import requests
import sqlite3

# Step 1: Download the SQL file
url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
sql_script = requests.get(url).text

# Step 2: Create the database
conn = sqlite3.connect("Chinook_2.db")
cursor = conn.cursor()
cursor.executescript(sql_script)
conn.commit()
conn.close()

Importing the API key from the env file

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # finds .env in project root
api_key = os.getenv("OPENAI_API_KEY")


Requesting the user to input the API Key (in case the env file doesn't contain the API key)

In [3]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ['OPENAI_API_KEY'] = getpass.getpass("Kindly input the OpenAI API Key")

Initiating the LLM Model Object

In [4]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider = "openai")

Creation of the System Prompt Template

In [ ]:
system_message_1 = '''You are an expert SQL generator.
   
   Given an input question, generate a syntactically correct {dialect} SQL query to answer the question.

   Only use the table names and columns as provided in the schema below. 
   If a table is named 'Employee', DO NOT USE 'Employees'. 
   If a table is named `OrderDetails`, DO NOT use `OrderDetail`.
   If a table is named 'Invoice', DO NOT use 'Invoices'
   Always use the table names and column names EXACTLY as shown below.

   Never query for all columns (no SELECT *). Select only the relevant columns based on the question.
   Unless the user asks for more, limit your responses to the at most {top_k} columns. You can sort the results using a relevant column if appropriate.
   
   ### Database schema
   {table_info}

   ### Example format
   ```sql
   select COLUMN 1 COLUMN 2 FROM TABLE 1 WHERE... LIMIT {top_k}   
  
  '''

In [ ]:
system_message_2 = '''You are an expert SQL generator.

Given an input question, generate a syntactically correct SQLite SQL query to answer the question.

Only use the table names and columns as provided below.  
Always use the table names and column names EXACTLY as shown.  
Do not use aliases unless necessary. Avoid SELECT * — select only relevant columns.  
Limit the number of results to {top_k} unless otherwise specified.

---

### TABLES AND COLUMNS

Table: Album  
Columns: AlbumId, Title, ArtistId

Table: Artist  
Columns: ArtistId, Name

Table: Customer  
Columns: CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId

Table: Employee  
Columns: EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email

Table: Genre  
Columns: GenreId, Name

Table: Invoice  
Columns: InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total

Table: InvoiceLine  
Columns: InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity

Table: MediaType  
Columns: MediaTypeId, Name

Table: Playlist  
Columns: PlaylistId, Name

Table: PlaylistTrack  
Columns: PlaylistId, TrackId

Table: Track  
Columns: TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice

---

### RELATIONSHIPS

Album.ArtistId → Artist.ArtistId  
Customer.SupportRepId → Employee.EmployeeId  
Employee.ReportsTo → Employee.EmployeeId  
Invoice.CustomerId → Customer.CustomerId  
InvoiceLine.TrackId → Track.TrackId  
InvoiceLine.InvoiceId → Invoice.InvoiceId  
PlaylistTrack.TrackId → Track.TrackId  
PlaylistTrack.PlaylistId → Playlist.PlaylistId  
Track.MediaTypeId → MediaType.MediaTypeId  
Track.GenreId → Genre.GenreId  
Track.AlbumId → Album.AlbumId

---

### Output Format (SQL)

```sql
SELECT Column1, Column2  
FROM Table1  
JOIN Table2 ON Table1.Col = Table2.Col  
WHERE condition  
LIMIT {top_k};'''


Creation of the full prompt template with system prompt template as one component

In [24]:
from langchain_core.prompts import ChatPromptTemplate
system_message = system_message_1
user_prompt ="Question: {input}"

query_prompt_template = ChatPromptTemplate(
                                          [("system", system_message),
                                           ("user", user_prompt)])


In [25]:
from typing_extensions import TypedDict
class State(TypedDict):
    user_question: str
    query: str
    result:str
    answer : str


Query Builder Function

In [26]:
from typing_extensions import Annotated

class StructuredOutput(TypedDict):
    query: Annotated[str,..., "sytactically valid SQL query"]

def query_builder(state:State) -> State:
    input_prompt = query_prompt_template.invoke({
        "dialect" : db.dialect,
        "table_info" : db.get_table_info(),
        "top_k": 10,
        "input": state['user_question'],
    })

    structured_llm = llm.with_structured_output(StructuredOutput)
    result = structured_llm.invoke(input_prompt)
    return{**state,
        "query":result['query']}


In [27]:
blank_state = {
    'user_question' : '' ,
    'query' : '',
    'result' : '',
    'answer' : ''
}
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook_2.db")
#blank_state['user_question'] = "How many employees are there ?"

#user_input = input("Ask your SQL question: ")
#blank_state['user_question'] = user_input


blank_state = query_builder(blank_state)


Query Executor Function

In [28]:
from langchain_community.tools.sql_database.tool import QuerySQLDatabaseTool

def query_executor(state:State) -> State:
    query_tool = QuerySQLDatabaseTool(db = db)
    result = query_tool.invoke(state['query'])
    return{**state, 
           'result': result} 

In [29]:
blank_state = query_executor(blank_state)
print(blank_state)

{'user_question': '', 'query': 'SELECT AlbumId, Title, ArtistId FROM Album LIMIT 10', 'result': "[(1, 'For Those About To Rock We Salute You', 1), (2, 'Balls to the Wall', 2), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3), (6, 'Jagged Little Pill', 4), (7, 'Facelift', 5), (8, 'Warner 25 Anos', 6), (9, 'Plays Metallica By Four Cellos', 7), (10, 'Audioslave', 8)]", 'answer': ''}


Answer Generator Function (Natural Language)

In [30]:
def generate_answer(state:State) -> State:
    prompt_for_nl_answer = ('Given the below user question, the generated query and the result, generate a natural language answer for the user.\n\n",'
                                   f"Question:{state['user_question']}",
                                   f"Query: {state['query']}",
                                   f"Result : {state['result']}")
    
    response_message = llm.invoke(prompt_for_nl_answer)
    return{**state,
           'answer':response_message.content}                             

In [31]:
blank_state = generate_answer(blank_state)
print(blank_state)

{'user_question': '', 'query': 'SELECT AlbumId, Title, ArtistId FROM Album LIMIT 10', 'result': "[(1, 'For Those About To Rock We Salute You', 1), (2, 'Balls to the Wall', 2), (3, 'Restless and Wild', 2), (4, 'Let There Be Rock', 1), (5, 'Big Ones', 3), (6, 'Jagged Little Pill', 4), (7, 'Facelift', 5), (8, 'Warner 25 Anos', 6), (9, 'Plays Metallica By Four Cellos', 7), (10, 'Audioslave', 8)]", 'answer': 'Here are the first 10 albums from our collection:\n\n1. **For Those About To Rock We Salute You** by Artist ID 1\n2. **Balls to the Wall** by Artist ID 2\n3. **Restless and Wild** by Artist ID 2\n4. **Let There Be Rock** by Artist ID 1\n5. **Big Ones** by Artist ID 3\n6. **Jagged Little Pill** by Artist ID 4\n7. **Facelift** by Artist ID 5\n8. **Warner 25 Anos** by Artist ID 6\n9. **Plays Metallica By Four Cellos** by Artist ID 7\n10. **Audioslave** by Artist ID 8\n\nIf you have any questions about these albums or need more information, feel free to ask!'}


Graph Building to combine all the functions in a sequence

In [32]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(State).add_sequence([query_builder, query_executor, generate_answer])

graph_builder.add_edge(START, "query_builder")

graph = graph_builder.compile()

Graph Execution : This executes all the nodes (functions) in a sequence. The state variable gets passed from one node to the next.

In [33]:
result = graph.invoke({"user_question" : "How many employess are there ?"})
print(result)                      

{'user_question': 'How many employess are there ?', 'query': 'SELECT COUNT(EmployeeId) AS TotalEmployees FROM Employee;', 'result': '[(8,)]', 'answer': 'There are a total of 8 employees.'}


In [34]:
employee_count_test = graph.invoke({"user_question" : "How many employees are there ?"})
print(employee_count_test)  

{'user_question': 'How many employees are there ?', 'query': 'SELECT COUNT(EmployeeId) as EmployeeCount FROM Employee;', 'result': '[(8,)]', 'answer': 'There are a total of 8 employees.'}


In [35]:
for step in graph.stream(
    {"user_question": "How many employees are there?"}, stream_mode="updates"
):
    print(step)

{'query_builder': {'user_question': 'How many employees are there?', 'query': 'SELECT COUNT(*) AS EmployeeCount FROM Employee'}}
{'query_executor': {'user_question': 'How many employees are there?', 'query': 'SELECT COUNT(*) AS EmployeeCount FROM Employee', 'result': '[(8,)]'}}
{'generate_answer': {'user_question': 'How many employees are there?', 'query': 'SELECT COUNT(*) AS EmployeeCount FROM Employee', 'result': '[(8,)]', 'answer': 'There are 8 employees.'}}


In [36]:
customer_count_test = graph.invoke({"user_question" : "How many customers are there ?"})
print(customer_count_test)  

{'user_question': 'How many customers are there ?', 'query': 'SELECT COUNT(CustomerId) AS TotalCustomers FROM Customer;', 'result': '[(59,)]', 'answer': 'There are a total of 59 customers.'}


In [37]:
album_count_test = graph.invoke({"user_question" : "How many albums are there ?"})
print(album_count_test) 

{'user_question': 'How many albums are there ?', 'query': 'SELECT COUNT(AlbumId) AS AlbumCount FROM Album;', 'result': '[(347,)]', 'answer': 'There are a total of 347 albums.'}


In [38]:
from run_graph_tests import run_graph_tests

# Assuming `graph` is already defined above this line
if __name__ == "__main__":
    run_graph_tests(graph)


✅ Test results saved to: test_results_20250924_0109.xlsx


Executing the same with an Agentic Architecture